In [4]:
import sys
!{sys.executable} -m pip install openai-whisper jiwer

  Using cached openai_whisper-20250625-py3-none-any.whl
  Using cached jiwer-4.0.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached more_itertools-11.1.0-py3-none-any.whl.metadata (41 kB)
  Using cached tiktoken-0.13.0-cp313-cp313-win_amd64.whl.metadata (6.8 kB)
  Using cached rapidfuzz-3.14.5-cp313-cp313-win_amd64.whl.metadata (12 kB)
Using cached jiwer-4.0.0-py3-none-any.whl (23 kB)
Using cached rapidfuzz-3.14.5-cp313-cp313-win_amd64.whl (1.5 MB)
Using cached more_itertools-11.1.0-py3-none-any.whl (72 kB)
Using cached tiktoken-0.13.0-cp313-cp313-win_amd64.whl (874 kB)

   ---------------------------------------- 0/5 [rapidfuzz]
   ---------------------------------------- 0/5 [rapidfuzz]
   ---------------------------------------- 0/5 [rapidfuzz]
   ---------------------------------------- 0/5 [rapidfuzz]
   -------- ------------------------------- 1/5 [more-itertools]
   ---------------- ----------------------- 2/5 [tiktoken]
   ------------------------ --------------- 3/5 [jiwer]
 


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import os
os.chdir("C:/Users/Lenovo/Desktop/Agai Project")  # use whichever path actually has your data/ folder
print(os.getcwd())

C:\Users\Lenovo\Desktop\Agai Project


In [8]:
print(os.path.exists("data/raw/audio_samples"))

True


In [10]:
import os
import pandas as pd

emotion_map = {
    '01': 'neutral', '02': 'calm', '03': 'happy', '04': 'sad',
    '05': 'angry', '06': 'fearful', '07': 'disgust', '08': 'surprised'
}

confidence_map = {
    'calm': 'confident', 'neutral': 'confident', 'happy': 'confident',
    'sad': 'nervous', 'fearful': 'nervous', 'disgust': 'nervous', 'angry': 'nervous',
    'surprised': 'neutral'
}

records = []
base_path = "data/raw/audio_samples"

for actor_folder in os.listdir(base_path):
    actor_path = os.path.join(base_path, actor_folder)
    if not os.path.isdir(actor_path):
        continue
    for fname in os.listdir(actor_path):
        if not fname.endswith('.wav'):
            continue
        parts = fname.replace('.wav', '').split('-')
        emotion_code = parts[2]
        records.append({
            'file_path': os.path.join(actor_path, fname),
            'actor': actor_folder,
            'emotion': emotion_map.get(emotion_code, 'unknown'),
            'intensity': 'strong' if parts[3] == '02' else 'normal'
        })

audio_index = pd.DataFrame(records)
audio_index['confidence_label'] = audio_index['emotion'].map(confidence_map)

print(audio_index.shape)

(1440, 5)


In [11]:
import whisper
from jiwer import wer
import random
import pandas as pd

model = whisper.load_model("base")

statement_map = {
    '01': 'Kids are talking by the door',
    '02': 'Dogs are sitting by the door'
}

def get_ground_truth(filename):
    parts = filename.replace('.wav', '').split('-')
    statement_code = parts[4]
    return statement_map.get(statement_code, None)

sample_files = audio_index.sample(50, random_state=42)

results = []
for idx, row in sample_files.iterrows():
    fname = row['file_path'].split('/')[-1].split('\\')[-1]
    ground_truth = get_ground_truth(fname)
    if ground_truth is None:
        continue

    transcription = model.transcribe(row['file_path'])['text'].strip()
    error_rate = wer(ground_truth.lower(), transcription.lower())
    accuracy = (1 - error_rate) * 100

    results.append({
        'file_path': row['file_path'],
        'ground_truth': ground_truth,
        'predicted': transcription,
        'accuracy': accuracy
    })

results_df = pd.DataFrame(results)
print("Average accuracy:", results_df['accuracy'].mean())
results_df.head(10)

c:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Average accuracy: 83.33333333333336


,file_path,ground_truth,predicted,accuracy
0,data/raw/audio_samples\Actor_03\03-01-07-02-01...,Kids are talking by the door,Kids are talking by the door.,83.333333
1,data/raw/audio_samples\Actor_11\03-01-02-01-01...,Kids are talking by the door,Kids are talking by the door.,83.333333
2,data/raw/audio_samples\Actor_10\03-01-02-02-01...,Kids are talking by the door,Kids are talking by the door.,83.333333
3,data/raw/audio_samples\Actor_02\03-01-02-01-01...,Kids are talking by the door,Kids are talking by the door.,83.333333
4,data/raw/audio_samples\Actor_11\03-01-05-01-01...,Kids are talking by the door,Kids are talking by the door.,83.333333
5,data/raw/audio_samples\Actor_17\03-01-02-01-01...,Kids are talking by the door,Kids are talking by the door.,83.333333
6,data/raw/audio_samples\Actor_24\03-01-02-02-02...,Dogs are sitting by the door,Dogs are sitting by the door.,83.333333
7,data/raw/audio_samples\Actor_24\03-01-06-02-02...,Dogs are sitting by the door,Dogs are sitting by the door!,83.333333
8,data/raw/audio_samples\Actor_20\03-01-08-02-02...,Dogs are sitting by the door,Dogs are sitting by the door.,83.333333
9,data/raw/audio_samples\Actor_05\03-01-02-01-01...,Kids are talking by the door,Kids are talking by the door.,83.333333


In [14]:
import re

def normalize_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)  # remove all punctuation
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Re-run accuracy calculation with normalization applied
def evaluate_model(model, sample_files):
    results = []
    for idx, row in sample_files.iterrows():
        fname = row['file_path'].split('/')[-1].split('\\')[-1]
        ground_truth = get_ground_truth(fname)
        if ground_truth is None:
            continue

        transcription = model.transcribe(row['file_path'])['text'].strip()
        
        gt_norm = normalize_text(ground_truth)
        pred_norm = normalize_text(transcription)
        
        error_rate = wer(gt_norm, pred_norm)
        accuracy = (1 - error_rate) * 100

        results.append({
            'ground_truth': ground_truth,
            'predicted': transcription,
            'accuracy': accuracy
        })
    return pd.DataFrame(results)

results_df_small_normalized = evaluate_model(model, sample_files)
print("Average accuracy (small, normalized):", results_df_small_normalized['accuracy'].mean())

c:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Average accuracy (small, normalized): 99.33333333333334


In [15]:
full_results = evaluate_model(model, audio_index)
print("Full dataset average accuracy:", full_results['accuracy'].mean())
print("Number of files evaluated:", len(full_results))

KeyboardInterrupt: 

In [16]:
from tqdm import tqdm

def evaluate_model(model, sample_files):
    results = []
    for idx, row in tqdm(sample_files.iterrows(), total=len(sample_files)):
        fname = row['file_path'].split('/')[-1].split('\\')[-1]
        ground_truth = get_ground_truth(fname)
        if ground_truth is None:
            continue

        transcription = model.transcribe(row['file_path'])['text'].strip()
        gt_norm = normalize_text(ground_truth)
        pred_norm = normalize_text(transcription)
        error_rate = wer(gt_norm, pred_norm)
        accuracy = (1 - error_rate) * 100

        results.append({
            'ground_truth': ground_truth,
            'predicted': transcription,
            'accuracy': accuracy
        })
    return pd.DataFrame(results)

full_results = evaluate_model(model, audio_index)
print("Full dataset average accuracy:", full_results['accuracy'].mean())

 19%|█▉        | 275/1440 [54:54<3:52:35, 11.98s/it]  


KeyboardInterrupt: 

In [17]:
sample_files_large = audio_index.sample(200, random_state=42)
full_results = evaluate_model(model, sample_files_large)
print("200-sample average accuracy:", full_results['accuracy'].mean())

100%|██████████| 200/200 [28:55<00:00,  8.68s/it]  

200-sample average accuracy: 99.58333333333334


In [18]:
full_results.to_csv("data/processed/audio/whisper_accuracy_results_200sample.csv", index=False)
print("Saved")

Saved


In [20]:
import os

real_speech_folder = "data/raw/real_speech_samples"  # adjust if you named it differently

files = os.listdir(real_speech_folder)
print(files)

['Describe a time you faced a challenge at work.mp3.mp3', 'Describe_your ideal work day.mp3.mp3', 'How do you align with our company values.mp3.mp3', 'How do you motivate others in a leadership role.mp3.mp3', 'What are your strengths and weaknesses.mp3.mp3', 'What do you want to work at our company.mp3.mp3', 'What makes a good leader in your opinion.mp3.mp3', 'What role do you usually play in group settings.mp3.mp3', 'What tools or methods help you stay organized.mp3.mp3', 'Where do you see yourself in five years.mp3.mp3']


In [24]:
ground_truth_real = {
    "Describe a time you faced a challenge at work.mp3.mp3":
        "Um, once I had to complete a project within a given, uh, within a, like, a very short deadline. Uh, at first I w- I was a little stressed, um, but then I broke the work into smaller tasks, uh, and then focus on them one by one. Uh, I also worked with my teammates to divide the work. Um, in the end, we completed it on time, and, uh, yeah, it taught me to stay calm under pressure, and it also just, uh, teach me how to deal with, uh, that pressure. Yeah.",

    "What are your strengths and weaknesses.mp3.mp3":
        "Um, I'd s- uh, say my biggest strength is that, uh, I'm a quick learner. Um, uh, if I, uh, don't know something, I try to learn it instead of giving up. Um, my biggest, uh, weakness is probably, uh, that I sometimes overthink my la-- my work and spend too much time on small things like, um, like ma-making improvements in them. So, uh, like, uh, but, but I'm working on managing, m-managing my time better to be a better individual and, uh, managing my work.",

    "Where do you see yourself in five years.mp3.mp3":
        "Uh, in five years, uh, I see myself, um, like, uh, as a skilled and confident professional. Uh, I want to have a good experience, handle bigger responsibilities, and hopefully, uh, you know, uh, lead some projects. Uh, most importantly, I want to keep learning and, uh, growing with the company itself.",

    "How do you motivate others in a leadership role.mp3.mp3":
        "Um, I think, uh, I motivate people by supporting them and keeping the environment positive. Um, if someone is struggling, I try to understand the problem and, uh, help them out. And I also appreciate people when they do something well. Uh, I think that, uh, really motivates a team.",

    "What do you want to work at our company.mp3.mp3":
        "Um, uh, uh, I want to work here because I feel this company would give me a good opportunity to learn and grow. Uh, I'd also get to work on real projects and, uh, you know, uh, from-- learn from experienced people. So yeah, I think it would be a great place for me to start and build my career.",

    "Describe_your ideal work day.mp3.mp3":
        "Um, my ideal workday is one where I know my priorities, uh, but still have room to learn something new. Um, uh, like starting, uh, by planning my task, then focusing on one thing at a time, uh, without too, uh, without too many distractions. I enjoy working with people, um, uh, sharing ideas and, uh, asking questions whenever I am stuck, because I believe collaboration, um, leads to better results. At the end of the day, I like looking back at what I've completed and what I learned. Even if the day is busy, if I've made progress and improved my skills, uh, I consider it as a successful day.",

    "What role do you usually play in group settings.mp3.mp3":
        "Uh, I usually become the person who keeps the group organized. Um, I naturally like dividing tasks, checking the deadlines, uh, and making sure everyone is on the same page. Uh, at the same time, I also enjoy listening to everyone's ideas, um, because good solutions often, uh, uh, come from different perspectives. Uh, uh, like, uh, like if somebody, uh, needs help with their part, I'm happy to support them. Uh, I don't always have to lead, but, uh, uh, I like making sure the team works, uh, smoothly and, uh, like, uh, everyone feels involved.",

    "What tools or methods help you stay organized.mp3.mp3":
        "Um, I, I prefer keeping things simple. I use Google Calendar to keep track of important, uh, dates and, uh, uh, well, uh, deadlines. And I maintain a to-do list on my phone where I prioritize tasks for the day. Uh, for bigger projects, I break into, uh, smaller goals because it makes everything feel more manageable. Um, I also review my progress at end of the, uh, day, uh, so like, uh, I know what needs attention next. This habit helps me stay consistent, um, without feeling overwhelmed.",

    "What makes a good leader in your opinion.mp3.mp3":
        "Um, uh, for me, a good leader is someone, uh, who leads by example instead of, um, uh, just giving instructions. A strong leader communicates clearly, uh, uh, listens to different opinions, and creates an environment, uh, where everyone feels comfortable sharing ideas. They stay calm under pressure, um, uh, take responsibility when things go wrong, and appreciate the team's efforts when things, uh, also go well. Uh, uh, I believe people naturally trust leaders who are supportive, respectful, and willing to learn, uh, alongside their team.",

    "How do you align with our company values.mp3.mp3":
        "Uh, uh, what attracts me most is the focus on learning, uh, teamwork, and continuous improvement. Uh, uh, I, uh, uh, I genuinely enjoy learning new skills, uh, and adapting to new challenges, uh, whether it's through projects, internships, or personal practice. Um, uh, I also believe in being de-dependable, uh, uh, communicating honestly, and contributing positively to the team. Uh, uh, like, uh, I'm still, uh, early in my career, so my goal is to keep improving every day, uh, while delivering my best work. I think that mindset, uh, matches a company, uh, that values growth, uh, collaboration, and, uh, responsibilities.",
}

print("Total ground truth entries:", len(ground_truth_real))

Total ground truth entries: 10


In [25]:
import os

real_speech_folder = "data/raw/real_speech_samples"  # confirm this matches your actual folder name

results = []
for fname, ground_truth in ground_truth_real.items():
    file_path = os.path.join(real_speech_folder, fname)
    transcription = model.transcribe(file_path)['text'].strip()

    gt_norm = normalize_text(ground_truth)
    pred_norm = normalize_text(transcription)
    error_rate = wer(gt_norm, pred_norm)
    accuracy = (1 - error_rate) * 100

    results.append({
        'question': fname,
        'ground_truth': ground_truth,
        'predicted': transcription,
        'accuracy': accuracy
    })

real_results_df = pd.DataFrame(results)
print(real_results_df[['question', 'accuracy']])
print("\nAverage accuracy on real speech (10 samples):", real_results_df['accuracy'].mean())

c:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


                                            question   accuracy
0  Describe a time you faced a challenge at work....  78.723404
1     What are your strengths and weaknesses.mp3.mp3  74.712644
2    Where do you see yourself in five years.mp3.mp3  76.923077
3  How do you motivate others in a leadership rol...  88.000000
4    What do you want to work at our company.mp3.mp3  83.333333
5               Describe_your ideal work day.mp3.mp3  82.882883
6  What role do you usually play in group setting...  76.288660
7  What tools or methods help you stay organized....  82.022472
8   What makes a good leader in your opinion.mp3.mp3  82.758621
9   How do you align with our company values.mp3.mp3  74.757282

Average accuracy on real speech (10 samples): 80.04023750197533


In [26]:
for idx, row in real_results_df.iterrows():
    print(f"\n{'='*70}")
    print(f"Question: {row['question']}")
    print(f"GROUND TRUTH:\n{row['ground_truth']}")
    print(f"\nPREDICTED:\n{row['predicted']}")
    print(f"\nAccuracy: {row['accuracy']:.1f}%")


Question: Describe a time you faced a challenge at work.mp3.mp3
GROUND TRUTH:
Um, once I had to complete a project within a given, uh, within a, like, a very short deadline. Uh, at first I w- I was a little stressed, um, but then I broke the work into smaller tasks, uh, and then focus on them one by one. Uh, I also worked with my teammates to divide the work. Um, in the end, we completed it on time, and, uh, yeah, it taught me to stay calm under pressure, and it also just, uh, teach me how to deal with, uh, that pressure. Yeah.

PREDICTED:
Once I had to complete a project within a very short deadline, at first I was a little stressed but then I broke the work into smaller tasks and then focused on them one by one. I also worked with my teammates to divide the work. In the end we completed it on time and it taught me to stay calm under pressure and it also just teach me how to deal with that pressure.

Accuracy: 78.7%

Question: What are your strengths and weaknesses.mp3.mp3
GROUND TRU

In [27]:
def normalize_content_only(text):
    text = text.lower()
    # Remove common filler words before comparing
    fillers = r'\b(um+|uh+|like|you know|so yeah)\b'
    text = re.sub(fillers, '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [29]:
import re

def normalize_content_only(text):
    text = text.lower()
    fillers = r'\b(um+|uh+|like|you know|yeah|so yeah)\b'
    text = re.sub(fillers, '', text)
    text = re.sub(r'-+', ' ', text)          # remove stutter dashes like "w- uh, was"
    text = re.sub(r'[^\w\s]', '', text)       # remove remaining punctuation
    text = re.sub(r'\s+', ' ', text).strip()
    return text

results_content = []
for fname, ground_truth in ground_truth_real.items():
    file_path = os.path.join(real_speech_folder, fname)
    transcription = model.transcribe(file_path)['text'].strip()

    gt_content = normalize_content_only(ground_truth)
    pred_content = normalize_content_only(transcription)
    error_rate = wer(gt_content, pred_content)
    accuracy = (1 - error_rate) * 100

    results_content.append({'question': fname, 'content_accuracy': accuracy})

content_df = pd.DataFrame(results_content)
print(content_df)
print("\nAverage CONTENT accuracy:", content_df['content_accuracy'].mean())

c:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


                                            question  content_accuracy
0  Describe a time you faced a challenge at work....         91.358025
1     What are your strengths and weaknesses.mp3.mp3         89.041096
2    Where do you see yourself in five years.mp3.mp3        100.000000
3  How do you motivate others in a leadership rol...        100.000000
4    What do you want to work at our company.mp3.mp3         96.000000
5               Describe_your ideal work day.mp3.mp3         91.752577
6  What role do you usually play in group setting...         96.000000
7  What tools or methods help you stay organized....         96.103896
8   What makes a good leader in your opinion.mp3.mp3         98.630137
9   How do you align with our company values.mp3.mp3         96.296296

Average CONTENT accuracy: 95.51820272878504
